# Milestone 2 — Transformer-only against Diffusion

**Optional extension:** *Transformer-only vs diffusion-based generation* (and the BERT-conditioned arm).

Three ways to turn the **same ten captions** into pictures:

| Arm | Encoder | How the image is made | Size |
|---|---|---|---|
| **SD 1.5** | CLIP ViT-L/14 | start from noise, denoise ~20–30 times — continuous latents | 512 |
| **aMUSEd** | CLIP ViT-L/14 | chop into discrete tiles, mask them all, fill in the blanks over ~12 rounds | 256 |
| **LDM-BERT** | BERT | latent diffusion, conditioned on BERT instead of CLIP | 256 |

**SD 1.5 against aMUSEd is the cleanest experiment in the milestone**: both use the *same* CLIP
ViT-L/14 text encoder, so the encoder is held fixed and the only variable is the generative mechanism.
aMUSEd has no noise, no noise schedule and no samplers at all — everything the previous notebook spent
its time tuning simply does not exist here.

**LDM-BERT changes the encoder instead**, which is why it is read as a separate axis rather than as a
fourth point on the same line.

### What does not transfer

The fixed-latent protocol. aMUSEd has no starting noise to hold still — its randomness is in which
tokens it commits to first — so the arms share the **prompts and the seed**, not the latents.
Within-arm comparisons were the previous notebook's job; this one compares across mechanisms, where
only the prompts can be held equal.

Each arm is freed before the next loads. 8 GB cannot hold two.

---
## 0. Kaggle setup

**Skipped entirely off Kaggle**, so this notebook still runs unchanged on a local machine.

On Kaggle it finds whichever attached dataset holds milestone 1's `processed/` output, copies it
somewhere writable, and points `config.py` at it. The copy is not optional: `/kaggle/input` is
read-only and `cfg.ensure_dirs()` creates directories under `DATA_DIR`.

This notebook also downloads ~7 GB of aMUSEd and LDM-BERT weights, so it needs a session with the
internet toggle **on**, or those weights attached as a second dataset with `HF_HOME` pointed at it.

This cell must stay **above the import cell** — `config.py` resolves every path once, when it is
first imported, and setting the variables afterwards has no effect.

In [ ]:
# --- Kaggle paths (no-op anywhere else) ------------------------------------
import os
from pathlib import Path

if Path("/kaggle/input").is_dir():
    import shutil

    attached = sorted(Path("/kaggle/input").iterdir())
    source = next((d for d in attached if (d / "processed" / "pairs.parquet").is_file()), None)
    if source is None:
        raise RuntimeError(
            "No attached dataset contains processed/pairs.parquet -- attach milestone 1's data/ "
            f"directory. Saw: {[d.name for d in attached]}"
        )

    work = Path("/kaggle/working/data")
    if not (work / "processed").is_dir():
        shutil.copytree(source / "processed", work / "processed")

    os.environ["T2I_DATA_DIR"] = str(work)
    os.environ["T2I_OUTPUT_DIR"] = "/kaggle/working/outputs"
    print(f"Kaggle: data from {source.name}, outputs to /kaggle/working/outputs")
else:
    print("Not on Kaggle -- using the repo's own data/ and outputs/ directories.")

In [ ]:
# --- make src/ importable --------------------------------------------------
import sys
from pathlib import Path


def _find_src():
    roots = [Path.cwd(), *Path.cwd().parents]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        roots += sorted(kaggle_input.iterdir())
    for root in roots:
        if (root / "src" / "config.py").is_file():
            return root / "src"
    raise RuntimeError("could not find src/config.py -- is this running inside the repo?")


SRC = _find_src()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"modules imported from: {SRC}")

In [ ]:
import time, warnings

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import transformers

warnings.filterwarnings("ignore", category=FutureWarning)
transformers.logging.set_verbosity_error()

import config as cfg
import utils
from modules import arms, clip_embeddings, prompts, recording, sanity_checks, sd_pipeline, storage

cfg.ensure_dirs()
utils.set_seed()

manifest = storage.read_manifest()
picks = prompts.fixed_prompts(manifest)
PROMPTS = picks["caption"].tolist()
REAL_IMAGES = [cfg.PROJECT_ROOT / p for p in picks["image_512_path"]]

clip_tokenizer, clip_joint, clip_processor = clip_embeddings.load_joint()

print(f"{len(PROMPTS)} prompts, seed {cfg.SEED}")
print(f"start: {sd_pipeline.vram_report()}")

## 1. SD 1.5 — the reference arm

The baseline from the sweep notebook, at DPMSolver++ and 20 steps.

In [ ]:
panels, timings, notes = {}, {}, {}

pipe, repo, base_config = sd_pipeline.load_sd()
sd_pipeline.set_scheduler(pipe, "dpmpp", base_config=base_config)

t0 = time.time()
panels["SD 1.5 (CLIP, diffusion, 512)"] = arms.run_arm(
    pipe, PROMPTS, num_inference_steps=cfg.SAMPLERS["dpmpp"], guidance_scale=cfg.GUIDANCE,
    generator=torch.Generator(device="cpu").manual_seed(cfg.SEED))
timings["SD 1.5 (CLIP, diffusion, 512)"] = time.time() - t0
notes["SD 1.5 (CLIP, diffusion, 512)"] = f"{repo}, DPMSolver++ {cfg.SAMPLERS['dpmpp']} steps"

print(f"{timings['SD 1.5 (CLIP, diffusion, 512)']:.1f}s   peak {sd_pipeline.vram_report()['peak_gb']} GB")

## 2. aMUSEd — the same encoder, a different mechanism

No `num_inference_steps` in the diffusion sense: aMUSEd unmasks tokens over a fixed number of rounds.
`arms.run_arm` passes only the arguments this pipeline actually declares, so nothing from SD's
vocabulary leaks across.

In [ ]:
amused, spec = arms.load_arm("amused", previous=pipe)
del pipe

t0 = time.time()
label = f"aMUSEd (CLIP, masked transformer, {spec['size']})"
panels[label] = arms.run_arm(
    amused, PROMPTS, num_inference_steps=12, guidance_scale=cfg.GUIDANCE,
    generator=torch.Generator(device="cpu").manual_seed(cfg.SEED))
timings[label] = time.time() - t0
notes[label] = f"{spec['repo']}, 12 unmasking rounds, encoder: {spec['encoder']}"

print(f"{timings[label]:.1f}s   peak {sd_pipeline.vram_report()['peak_gb']} GB")

## 3. LDM-BERT — the BERT-conditioned arm

Latent diffusion like SD, but conditioned on BERT. This is milestone 2's BERT coverage without
training anything: a pretrained model that already does what a BERT→CLIP adapter would have to learn.

In [ ]:
ldm, spec = arms.load_arm("ldm_bert", previous=amused)
del amused

t0 = time.time()
label = f"LDM-BERT (BERT, diffusion, {spec['size']})"
panels[label] = arms.run_arm(
    ldm, PROMPTS, num_inference_steps=50, guidance_scale=cfg.GUIDANCE,
    generator=torch.Generator(device="cpu").manual_seed(cfg.SEED))
timings[label] = time.time() - t0
notes[label] = f"{spec['repo']}, 50 steps, encoder: {spec['encoder']}"

sd_pipeline.free(ldm); del ldm
print(f"{timings[label]:.1f}s   released: {sd_pipeline.vram_report()}")

## 4. Scoring

One metric across all three: CLIP(text, generated), the same cosine the sweep used. It is scored in
CLIP ViT-B/32's joint space — a *different* CLIP from the ViT-L/14 that conditions SD and aMUSEd, so
the judge is not simply one of the contestants' own encoders.

Resolution does not bias it: CLIP resizes everything to 224 before looking, so the 256px arms are not
penalised for being small. What a bare cosine *does* miss is image quality — a sharp image and a
muddy one that both contain the right objects score alike. Read the grid alongside the table.

In [ ]:
rows = []
for label, images in panels.items():
    txt_gen, txt_real, gen_real = sanity_checks.clip_scores(
        PROMPTS, images, REAL_IMAGES, clip_tokenizer, clip_joint, clip_processor)
    rows.append({
        "arm": label,
        "clip_text_generated": round(float(txt_gen.mean()), 4),
        "std": round(float(txt_gen.std()), 4),
        "clip_generated_real": round(float(gen_real.mean()), 4),
        "seconds": round(timings[label], 1),
        "s_per_image": round(timings[label] / len(PROMPTS), 2),
        "size": images[0].size[0],
    })

results = pd.DataFrame(rows).sort_values("clip_text_generated", ascending=False)
reference = float(txt_real.mean())
print(f"real COCO photographs score {reference:.4f} -- the number to read these against\n")
display(results)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.0))
y = np.arange(len(results))
ax.barh(y, results["clip_text_generated"], xerr=results["std"], capsize=4)
ax.axvline(reference, ls="--", c="grey", label=f"real photos ({reference:.3f})")
ax.set_yticks(y); ax.set_yticklabels(results["arm"], fontsize=8); ax.invert_yaxis()
ax.set_xlabel("CLIP(text, generated)")
ax.set_title("Prompt adherence across generative mechanisms")
ax.legend(); ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
print(utils.save_fig(fig, "m2_arm_comparison"))
plt.show()

## 5. The pictures

One row per arm, the same prompts down the columns. Tiles are resized to a common width so the rows
line up — the 256px arms genuinely produce smaller images, which the table records.

In [ ]:
WIDTH = 256
normalised = {label: [img.resize((WIDTH, WIDTH)) for img in images]
              for label, images in panels.items()}

grid = recording.comparison_grid(normalised)
grid_path = cfg.SWEEP_DIR / "arm_comparison.png"
grid.save(grid_path)
print(utils.rel(grid_path))

for label, images in panels.items():
    print(recording.save_run(images, PROMPTS, recording.slugify(label))[-1])

grid

In [ ]:
summary = {
    "prompts": PROMPTS,
    "seed": cfg.SEED,
    "protocol_note": ("shared prompts and seed, not shared latents -- aMUSEd has no starting "
                      "noise, so the sweep's fixed-latent protocol does not transfer"),
    "scoring_note": "CLIP ViT-B/32 judges; SD and aMUSEd are conditioned on ViT-L/14",
    "reference_real_photos": round(reference, 4),
    "arms": {row["arm"]: {**{k: v for k, v in row.items() if k != "arm"},
                          "detail": notes[row["arm"]]} for row in rows},
}
print(utils.save_table("milestone_2_comparison", summary))
print()
print(f"best adherence: {results.iloc[0]['arm']}  ({results.iloc[0]['clip_text_generated']:.4f})")
print(f"fastest       : {results.sort_values('s_per_image').iloc[0]['arm']}  "
      f"({results['s_per_image'].min():.2f}s per image)")

### What this shows

The sweep notebook spent its time on samplers and noise schedules. aMUSEd has none of those, and still
turns captions into pictures from the same CLIP embedding — so the noise schedule is a property of
*one way* of generating images, not of text-to-image generation as such.

LDM-BERT swaps the encoder instead of the mechanism, which is the other half of milestone 2's BERT
coverage and needs no training to get.